# Preamble

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import warnings
import keysight.qcs as qcs
import numpy as np
import matplotlib.pyplot as plt
# import sys
# sys.path.append("..")

from keysight.qcs.experiments import ResonatorSpectroscopy, ResonatorSpectroscopy2D, DispersiveShift
from keysight.qcs.experiments import QubitSpectroscopy,RabiExperiment,T1Experiment,RamseyExperiment,IQDistributionExperiment
from keysight.qcs.analysis import FanoResonance, ComplexLorentzian

from preamble.pump_utils import *
from preamble.mapper_utils import *
from preamble.calibration_utils import *
from preamble.pca_gmm_utils import *
from preamble.fitter import *


us,ns,MHz,GHz=1e-6,1e-9,1e6,1e9
pi=np.pi

In [ ]:
cal_file='./config/260702_cal.json'
# set the qubit topology
qubits = qcs.Qudits([0,1,2])
n_qubits=len(qubits)

pairs = [(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)]
n_mq = len(pairs)

topology = qcs.QuditGraph(qubits, pairs)

# set the local oscillator frequency
LO_freq_readout =  6.1*GHz
LO_freq_qubit = 4.3*GHz

# load the calibration variables
variables_from_json = load_json(cal_file)
variables=make_qcs_var(
            variables_from_json=variables_from_json,
            qubits=qubits,
        )

mapper = qcs.ChannelMapper('localhost')
address = load_address("./config/channel_address.yaml")

channel_mapping(mapper,qubits,address)
set_lo_frequencies_RO(mapper,qubits,address,LO_freq_readout)
set_lo_frequencies_XY(mapper,qubits,address,LO_freq_qubit) 
# set_lo_frequencies_XY(mapper,qubits[0],address,4.1*GHz)

backend = qcs.HclBackend(mapper, fpga_postprocessing=True ,init_time=150*us)

cals = MyCalibrationSet(
        topology=topology,
        channels=mapper.channels,
        variables=variables)
cals.export_values(cal_file)

# shorten the variables name
c=cals.variables


print("Backend is ready:", backend.is_system_ready())


In [ ]:
# set the port of windfreak
# port='socket://163.152.38.107:4000' 
port='COM3'
pump = PumpController(port=port).connect()
pump.set(ch=1,freq=7.52*GHz, power=8.4)
pump.close_all()


# Abort program 

In [ ]:
current_id=backend.get_program_execution_history()[0]['accession_id']
backend.abort_program(current_id)

# Q0

## Two-tone spectroscopy (Overlapped)

In [ ]:
target_qubit=qubits[0]
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.sx_amp,0.05, index=range(n_qubits))

set_value_at(c.ro_dur,3*us, index=range(n_qubits))
set_value_at(c.ro_delay,-0.9*c.ro_dur.value, index=range(n_qubits))
set_value_at(c.acq_dur,c.ro_dur.value, index=range(n_qubits))
set_value_at(c.sx_dur,c.ro_dur.value, index=range(n_qubits))

f_start = 4.5 * GHz
f_end = 4.9 * GHz
n_steps = 101

frequencies=np.linspace(f_start, f_end, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(1000)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

cals.import_values('./config/temp.json')
two_tone.plot_iq(plot_type="linear")

In [ ]:
c.sx_freq[target_qubit.labels].value=4.832*GHz
cals.export_values(cal_file)

## Amplitude Rabi

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
amp_start = 0
amp_end = 1
n_steps = 51

amplitudes=np.linspace(amp_start, amp_end, n_steps)
amplitudes=qcs.Array('amplitudes',value=amplitudes, dtype=float)

##############################
amp_rabi = qcs.Program()
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_measurement(target_qubit, new_layer=True)

amp_rabi = qcs.LinkerPass(*cals.linkers.values()).apply(amp_rabi)

amp_rabi.n_shots(500)
amp_rabi.sweep(amplitudes, c.sx_amp[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
amp_rabi = qcs.Executor(backend).execute(amp_rabi)
pump.off(ch=1).close_all()


amp_rabi.plot_iq(plot_type="linear")

In [ ]:
_, halfpi_amp=amplitude_rabi_fit(amplitudes.value,np.abs(amp_rabi.get_iq_array(avg=True)[0]))

In [ ]:
c.sx_amp[target_qubit.labels].value=halfpi_amp
cals.export_values(cal_file)

## Two-tone spectroscopy (Pi pulse)

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
f_center = c.sx_freq[list(target_qubit.labels)].value
f_range = 200 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_gate(qcs.GATES.x90, target_qubit)

two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(500)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

two_tone.plot_iq(plot_type="linear")

In [ ]:
f_Q0=fit_lorentzian(frequencies.value, np.abs(two_tone.get_iq_array(avg=True)[0]))['f0']

In [ ]:
c.sx_freq[target_qubit.labels].value=f_Q0
cals.export_values(cal_file)

## Dispersive Shift

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 10 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
g_one_tone = qcs.Program()
g_one_tone.add_measurement(target_qubit, new_layer=True)

g_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(g_one_tone)

g_one_tone.n_shots(500)
g_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
g_one_tone = qcs.Executor(backend).execute(g_one_tone)
pump.off(ch=1).close_all()


# excited state preparation
##############################
e_one_tone = qcs.Program()
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)

e_one_tone.add_measurement(target_qubit, new_layer=True)

e_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(e_one_tone)

e_one_tone.n_shots(500)
e_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
e_one_tone = qcs.Executor(backend).execute(e_one_tone)
pump.off(ch=1).close_all()

##########
plt.plot(frequencies.value/GHz,np.abs(g_one_tone.get_iq_array(avg=True)[0]),label='ground')
plt.plot(frequencies.value/GHz,np.abs(e_one_tone.get_iq_array(avg=True)[0]),label='excited')
plt.xlabel('Frequency [GHz]')
plt.ylabel('Signal (arb.)')
plt.legend()
plt.show()

In [ ]:

out_0=fit_lorentzian(frequencies.value,np.abs(g_one_tone.get_iq_array(avg=True)[0]))
out_1=fit_lorentzian(frequencies.value,np.abs(e_one_tone.get_iq_array(avg=True)[0]))

print("chi=",round((out_0["f0"]-out_1["f0"])/MHz/2,3),"MHz")

## T1 experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 100*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

t1 = qcs.Program()
t1.add_gate(qcs.GATES.x90, target_qubit)
t1.add_gate(qcs.GATES.x90, target_qubit)

t1.add_measurement(target_qubit, new_layer=True,pre_delay=delay)

t1 = qcs.LinkerPass(*cals.linkers.values()).apply(t1)

t1.n_shots(2000)
t1.sweep(delays, delay)
##############################

pump.connect().set_on(ch=1)
t1 = qcs.Executor(backend).execute(t1)
pump.off(ch=1).close_all()

In [ ]:
t1_fit(delays.value,np.abs(t1.get_iq_array(avg=True))[0],t1_init=50*us)

## Ramsey experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[0]

detuning=0.2*MHz # artificial detuning using virtual Z
# Set the frequency range for sweeping
t_start = 0*us
t_end = 40*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)
phase=qcs.Scalar('phase',value=0,dtype=float)

ramsey = qcs.Program()
ramsey.add_gate(qcs.GATES.x90, target_qubit)
ramsey.add_parametric_gate(qcs.PAULIS.rz, phase,target_qubit,pre_delay=delay)
ramsey.add_gate(qcs.GATES.x90, target_qubit)

ramsey.add_measurement(target_qubit, new_layer=True)

ramsey = qcs.LinkerPass(*cals.linkers.values()).apply(ramsey)

ramsey.n_shots(1000)
ramsey.sweep([delays,2*np.pi*delays*detuning], [delay,phase])
# ramsey.sweep([delays], [delay])

##############################

pump.connect().set_on(ch=1)
ramsey = qcs.Executor(backend).execute(ramsey)
pump.off(ch=1).close_all()

In [ ]:
_,ramsey_f=t2_fit(delays.value,np.abs(ramsey.get_iq_array(avg=True))[0],T2_guess=13*us,delta_guess=detuning)

In [ ]:
c.sx_freq[target_qubit.labels].value += detuning-ramsey_f
cals.export_values(cal_file)

## Echo experiment

In [ ]:
target_qubit=qubits[0]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 60*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays/2, dtype=float)

##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

echo = qcs.Program()
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)

echo.add_measurement(target_qubit, new_layer=True)

echo = qcs.LinkerPass(*cals.linkers.values()).apply(echo)

echo.n_shots(1000)
echo.sweep(delays, delay)

##############################
delays=delays*2

pump.connect().set_on(ch=1)
echo = qcs.Executor(backend).execute(echo)
pump.off(ch=1).close_all()

In [ ]:
t2_fit(delays.value,np.abs(echo.get_iq_array(avg=True))[0],T2_guess=20*us,delta_guess=0)

# Q1

## Two-tone spectroscopy (Overlapped)

In [ ]:
target_qubit=qubits[1]
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.sx_amp,0.025, index=range(n_qubits))

set_value_at(c.ro_dur,3*us, index=range(n_qubits))
set_value_at(c.ro_delay,-0.9*c.ro_dur.value, index=range(n_qubits))
set_value_at(c.acq_dur,c.ro_dur.value, index=range(n_qubits))
set_value_at(c.sx_dur,c.ro_dur.value, index=range(n_qubits))

f_start = 4.4 * GHz
f_end = 4.7 * GHz
n_steps = 101

frequencies=np.linspace(f_start, f_end, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(1000)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

cals.import_values('./config/temp.json')
two_tone.plot_iq(plot_type="linear")

In [ ]:
c.sx_freq[target_qubit.labels].value=4.526*GHz
cals.export_values(cal_file)

## Amplitude Rabi

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[1]

# Set the frequency range for sweeping
amp_start = 0
amp_end = 1
n_steps = 51

amplitudes=np.linspace(amp_start, amp_end, n_steps)
amplitudes=qcs.Array('amplitudes',value=amplitudes, dtype=float)

##############################
amp_rabi = qcs.Program()
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_measurement(target_qubit, new_layer=True)

amp_rabi = qcs.LinkerPass(*cals.linkers.values()).apply(amp_rabi)

amp_rabi.n_shots(1000)
amp_rabi.sweep(amplitudes, c.sx_amp[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
amp_rabi = qcs.Executor(backend).execute(amp_rabi)
pump.off(ch=1).close_all()


amp_rabi.plot_iq(plot_type="linear")

In [ ]:
_, halfpi_amp=amplitude_rabi_fit(amplitudes.value,np.imag(amp_rabi.get_iq_array(avg=True)[0]))

In [ ]:
c.sx_amp[target_qubit.labels].value=halfpi_amp
cals.export_values(cal_file)

## Two-tone spectroscopy (Pi pulse)

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[1]

# Set the frequency range for sweeping
f_center = c.sx_freq[list(target_qubit.labels)].value
f_range = 150 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_gate(qcs.GATES.x90, target_qubit)

two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(500)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

two_tone.plot_iq(plot_type="linear")

In [ ]:
f_Q1=fit_lorentzian(frequencies.value, np.imag(two_tone.get_iq_array(avg=True)[0]))['f0']

In [ ]:
c.sx_freq[target_qubit.labels].value=f_Q1
cals.export_values(cal_file)

## Dispersive Shift

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[1]

# Set the frequency range for sweeping
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 10 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
g_one_tone = qcs.Program()
g_one_tone.add_measurement(target_qubit, new_layer=True)

g_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(g_one_tone)

g_one_tone.n_shots(500)
g_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
g_one_tone = qcs.Executor(backend).execute(g_one_tone)
pump.off(ch=1).close_all()


# excited state preparation
##############################
e_one_tone = qcs.Program()
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)

e_one_tone.add_measurement(target_qubit, new_layer=True)

e_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(e_one_tone)

e_one_tone.n_shots(500)
e_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
e_one_tone = qcs.Executor(backend).execute(e_one_tone)
pump.off(ch=1).close_all()

##########
plt.plot(frequencies.value/GHz,np.abs(g_one_tone.get_iq_array(avg=True)[0]),label='ground')
plt.plot(frequencies.value/GHz,np.abs(e_one_tone.get_iq_array(avg=True)[0]),label='excited')
plt.xlabel('Frequency [GHz]')
plt.ylabel('Signal (arb.)')
plt.legend()
plt.show()

In [ ]:

out_0=fit_lorentzian(frequencies.value,np.abs(g_one_tone.get_iq_array(avg=True)[0]))
out_1=fit_lorentzian(frequencies.value,np.abs(e_one_tone.get_iq_array(avg=True)[0]))

print("chi=",round((out_0["f0"]-out_1["f0"])/MHz/2,3),"MHz")

## T1 experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[1]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 100*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

t1 = qcs.Program()
t1.add_gate(qcs.GATES.x90, target_qubit)
t1.add_gate(qcs.GATES.x90, target_qubit)

t1.add_measurement(target_qubit, new_layer=True,pre_delay=delay)

t1 = qcs.LinkerPass(*cals.linkers.values()).apply(t1)

t1.n_shots(1000)
t1.sweep(delays, delay)
##############################

pump.connect().set_on(ch=1)
t1 = qcs.Executor(backend).execute(t1)
pump.off(ch=1).close_all()

In [ ]:
t1_fit(delays.value,np.imag(t1.get_iq_array(avg=True))[0],t1_init=50*us)

## Ramsey experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[1]

detuning=0.3*MHz # artificial detuning using virtual Z
# Set the frequency range for sweeping
t_start = 0*us
t_end = 30*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)
phase=qcs.Scalar('phase',value=0,dtype=float)

ramsey = qcs.Program()
ramsey.add_gate(qcs.GATES.x90, target_qubit)
ramsey.add_parametric_gate(qcs.PAULIS.rz, phase,target_qubit,pre_delay=delay)
ramsey.add_gate(qcs.GATES.x90, target_qubit)

ramsey.add_measurement(target_qubit, new_layer=True)

ramsey = qcs.LinkerPass(*cals.linkers.values()).apply(ramsey)

ramsey.n_shots(1000)
ramsey.sweep([delays,2*np.pi*delays*detuning], [delay,phase])
# ramsey.sweep([delays], [delay])

##############################

pump.connect().set_on(ch=1)
ramsey = qcs.Executor(backend).execute(ramsey)
pump.off(ch=1).close_all()

In [ ]:
_,ramsey_f=t2_fit(delays.value,np.imag(ramsey.get_iq_array(avg=True))[0],T2_guess=5*us,delta_guess=detuning)

In [ ]:
c.sx_freq[target_qubit.labels].value += detuning-ramsey_f
cals.export_values(cal_file)

## Echo experiment

In [ ]:
target_qubit=qubits[1]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 60*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays/2, dtype=float)

##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

echo = qcs.Program()
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)

echo.add_measurement(target_qubit, new_layer=True)

echo = qcs.LinkerPass(*cals.linkers.values()).apply(echo)

echo.n_shots(1000)
echo.sweep(delays, delay)

##############################
delays=delays*2

pump.connect().set_on(ch=1)
echo = qcs.Executor(backend).execute(echo)
pump.off(ch=1).close_all()

In [ ]:
t2_fit(delays.value,np.imag(echo.get_iq_array(avg=True))[0],T2_guess=20*us,delta_guess=0)

# Q2

## Two-tone spectroscopy (Overlapped)

In [ ]:
target_qubit=qubits[2]
cals.import_values(cal_file)
cals.export_values('./config/temp.json')

set_value_at(c.sx_amp,0.01, index=range(n_qubits))

set_value_at(c.ro_dur,3*us, index=range(n_qubits))
set_value_at(c.ro_delay,-0.9*c.ro_dur.value, index=range(n_qubits))
set_value_at(c.acq_dur,c.ro_dur.value, index=range(n_qubits))
set_value_at(c.sx_dur,c.ro_dur.value, index=range(n_qubits))

f_start = 4.4 * GHz
f_end = 4.65 * GHz
n_steps = 101

frequencies=np.linspace(f_start, f_end, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(1000)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

cals.import_values('./config/temp.json')
two_tone.plot_iq(plot_type="linear")

In [ ]:
c.sx_freq[target_qubit.labels].value=4.595*GHz
cals.export_values(cal_file)

## Amplitude Rabi

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[2]

# Set the frequency range for sweeping
amp_start = 0
amp_end = 1
n_steps = 51

amplitudes=np.linspace(amp_start, amp_end, n_steps)
amplitudes=qcs.Array('amplitudes',value=amplitudes, dtype=float)

##############################
amp_rabi = qcs.Program()
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_gate(qcs.GATES.x90, target_qubit)
amp_rabi.add_measurement(target_qubit, new_layer=True)

amp_rabi = qcs.LinkerPass(*cals.linkers.values()).apply(amp_rabi)

amp_rabi.n_shots(500)
amp_rabi.sweep(amplitudes, c.sx_amp[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
amp_rabi = qcs.Executor(backend).execute(amp_rabi)
pump.off(ch=1).close_all()


amp_rabi.plot_iq(plot_type="linear")

In [ ]:
_, halfpi_amp=amplitude_rabi_fit(amplitudes.value,np.real(amp_rabi.get_iq_array(avg=True)[0]))

In [ ]:
halfpi_amp

In [ ]:
c.sx_amp[target_qubit.labels].value=halfpi_amp
cals.export_values(cal_file)

In [ ]:
c.sx_amp.value

## Two-tone spectroscopy (Pi pulse)

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[2]

# Set the frequency range for sweeping
f_center = c.sx_freq[list(target_qubit.labels)].value
f_range = 100 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

##############################
two_tone = qcs.Program()
two_tone.add_gate(qcs.GATES.x90, target_qubit)
two_tone.add_gate(qcs.GATES.x90, target_qubit)

two_tone.add_measurement(target_qubit, new_layer=True)

two_tone = qcs.LinkerPass(*cals.linkers.values()).apply(two_tone)

two_tone.n_shots(500)
two_tone.sweep(frequencies, c.sx_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
two_tone = qcs.Executor(backend).execute(two_tone)
pump.off(ch=1).close_all()

two_tone.plot_iq(plot_type="linear")

In [ ]:
f_Q2=fit_lorentzian(frequencies.value, np.real(two_tone.get_iq_array(avg=True)[0]))['f0']

In [ ]:
c.sx_freq[target_qubit.labels].value=f_Q2
cals.export_values(cal_file)

## Dispersive Shift

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[2]

# Set the frequency range for sweeping
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 10 * MHz
n_steps = 51

frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)
frequencies=qcs.Array('frequencies',value=frequencies, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
g_one_tone = qcs.Program()
g_one_tone.add_measurement(target_qubit, new_layer=True)

g_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(g_one_tone)

g_one_tone.n_shots(500)
g_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
g_one_tone = qcs.Executor(backend).execute(g_one_tone)
pump.off(ch=1).close_all()


# excited state preparation
##############################
e_one_tone = qcs.Program()
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)
e_one_tone.add_gate(qcs.GATES.x90, target_qubit)

e_one_tone.add_measurement(target_qubit, new_layer=True)

e_one_tone = qcs.LinkerPass(*cals.linkers.values()).apply(e_one_tone)

e_one_tone.n_shots(500)
e_one_tone.sweep(frequencies, c.ro_freq[target_qubit.labels])
##############################

pump.connect().set_on(ch=1)
e_one_tone = qcs.Executor(backend).execute(e_one_tone)
pump.off(ch=1).close_all()

##########
plt.plot(frequencies.value/GHz,np.abs(g_one_tone.get_iq_array(avg=True)[0]),label='ground')
plt.plot(frequencies.value/GHz,np.abs(e_one_tone.get_iq_array(avg=True)[0]),label='excited')
plt.xlabel('Frequency [GHz]')
plt.ylabel('Signal (arb.)')
plt.legend()
plt.show()

In [ ]:

out_0=fit_lorentzian(frequencies.value,np.abs(g_one_tone.get_iq_array(avg=True)[0]))
out_1=fit_lorentzian(frequencies.value,np.abs(e_one_tone.get_iq_array(avg=True)[0]))

print("chi=",round((out_0["f0"]-out_1["f0"])/MHz/2,3),"MHz")

## T1 experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[2]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 200*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

t1 = qcs.Program()
t1.add_gate(qcs.GATES.x90, target_qubit)
t1.add_gate(qcs.GATES.x90, target_qubit)

t1.add_measurement(target_qubit, new_layer=True,pre_delay=delay)

t1 = qcs.LinkerPass(*cals.linkers.values()).apply(t1)

t1.n_shots(2000)
t1.sweep(delays, delay)
##############################

pump.connect().set_on(ch=1)
t1 = qcs.Executor(backend).execute(t1)
pump.off(ch=1).close_all()

In [ ]:
t1_fit(delays.value,np.real(t1.get_iq_array(avg=True))[0],t1_init=50*us)

## Ramsey experiment

In [ ]:
cals.import_values(cal_file)

In [ ]:
target_qubit=qubits[2]

detuning=1*MHz # artificial detuning using virtual Z
# Set the frequency range for sweeping
t_start = 0*us
t_end = 5*us
n_steps = 101

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays, dtype=float)

# set_lo_frequencies_XY(mapper,qubits,address,f_start-50*MHz) 

# ground state preparation
##############################
delay=qcs.Scalar('delay',value=0,dtype=float)
phase=qcs.Scalar('phase',value=0,dtype=float)

ramsey = qcs.Program()
ramsey.add_gate(qcs.GATES.x90, target_qubit)
ramsey.add_parametric_gate(qcs.PAULIS.rz, phase,target_qubit,pre_delay=delay)
ramsey.add_gate(qcs.GATES.x90, target_qubit)

ramsey.add_measurement(target_qubit, new_layer=True)

ramsey = qcs.LinkerPass(*cals.linkers.values()).apply(ramsey)

ramsey.n_shots(1000)
ramsey.sweep([delays,2*np.pi*delays*detuning], [delay,phase])
# ramsey.sweep([delays], [delay])

##############################

pump.connect().set_on(ch=1)
ramsey = qcs.Executor(backend).execute(ramsey)
pump.off(ch=1).close_all()

In [ ]:
_,ramsey_f=t2_fit(delays.value,np.real(ramsey.get_iq_array(avg=True))[0],T2_guess=3*us,delta_guess=detuning+0.1*MHz)

In [ ]:
c.sx_freq[target_qubit.labels].value += detuning-ramsey_f
cals.export_values(cal_file)

## Echo experiment

In [ ]:
target_qubit=qubits[2]

# Set the frequency range for sweeping
t_start = 0*us
t_end = 80*us
n_steps = 51

delays=np.linspace(t_start, t_end, n_steps)
delays=qcs.Array('delays',value=delays/2, dtype=float)

##############################
delay=qcs.Scalar('delay',value=0,dtype=float)

echo = qcs.Program()
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)
echo.add_gate(qcs.GATES.x90, target_qubit)
echo.add_gate(qcs.GATES.x90, target_qubit,pre_delay=delay)

echo.add_measurement(target_qubit, new_layer=True)

echo = qcs.LinkerPass(*cals.linkers.values()).apply(echo)

echo.n_shots(1000)
echo.sweep(delays, delay)

##############################
delays=delays*2

pump.connect().set_on(ch=1)
echo = qcs.Executor(backend).execute(echo)
pump.off(ch=1).close_all()

In [ ]:
t2_fit(delays.value,np.real(echo.get_iq_array(avg=True))[0],T2_guess=20*us,delta_guess=0)

# Make cal_backup

In [ ]:
cals.export_values(cal_file.removesuffix('.json')+'_backup.json')